# Three-epoch baseline pilot
Use updated BTP-code-pilot.zip. Download the complete generated archive before stopping.

In [ ]:
from pathlib import Path
import subprocess, sys, torch
CODE = Path('/kaggle/working/BTP')
DATA = Path('/kaggle/input/datasets/arnavnigamd/btp-data/public252')
assert CODE.joinpath('run_public252.py').is_file(), 'Extract the updated BTP-code-pilot.zip first.'
assert '--stop-after-epoch' in CODE.joinpath('run_public252.py').read_text(), 'Use the updated pilot code ZIP.'
assert torch.cuda.is_available(), 'Enable the GPU before starting.'
print(torch.cuda.get_device_name(0))
subprocess.run([sys.executable, 'run_public252.py', '--root', str(DATA), '--epochs', '500', '--stop-after-epoch', '3'], cwd=CODE, check=True)


In [ ]:
from datetime import datetime
import shutil, zipfile
from IPython.display import display, FileLink
NAME = 'baseline_pilot_' + datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT = Path('/kaggle/working/baseline_runs')
RUN = OUTPUT / NAME
try:
    subprocess.run([sys.executable, '-u', 'run_public252.py', '--root', str(DATA),
        '--train', '--epochs', '500', '--stop-after-epoch', '3', '--batch-size', '1',
        '--workers', '0', '--output', str(OUTPUT), '--name', NAME], cwd=CODE, check=True)
finally:
    if RUN.exists():
        archive = shutil.make_archive(str(Path('/kaggle/working') / NAME), 'zip', root_dir=OUTPUT, base_dir=NAME)
        with zipfile.ZipFile(archive) as z:
            assert z.testzip() is None, 'Archive integrity check failed'
            required = [NAME + '/model/' + f for f in ['last.pt', 'best_iou.pth', 'best_psnr.pth']]
            missing = [f for f in required if f not in z.namelist()]
        print('Archive:', archive, 'bytes:', Path(archive).stat().st_size)
        print('Missing checkpoint files:', missing)
        display(FileLink(archive))
        print('Download this archive before stopping the session. If checkpoint files are missing, report the training error.')


In [ ]:
import json
for path in sorted((RUN / 'result').glob('*_training.json')):
    print(json.dumps(json.loads(path.read_text()), indent=2))
print('Download the complete archive, then share these metrics and the per-class segmentation CSVs.')
